# Extended Data Figure 4: Benefit-to-cost ratio

Mean long-run cooperation $\bar\rho$ as a function of $b/c$ without inertia, with full one-step inertia and with the
selected multi-step depths ($e_1=e_2=0.02$). Each point averages eight independent scrambled-Sobol samples of 512
area-uniform initial compositions (seeds 2026–2033); shading shows $\pm$ one standard error across the eight sample
means. Line style marks the $b/c$ interval in which a pure DISC population excludes rare ALLD and is not invaded by
rare ALLC ($R_L<b/c<R_U$). Run from this directory. Saved data are used by default; the scan is regenerated with
`python ../engine/scan.py ratio 0 1` followed by `python ../engine/postprocess.py ratio`.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.interpolate import PchipInterpolator
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str((ROOT / '..' / 'engine').resolve()))
import analysis
DATA = ROOT / 'data' / 'extended_data_figure4'
FIGURES = ROOT / 'figures' / 'extended_data_figure4'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
E1 = E2 = .02
N_SAMPLES, N_INITIAL = 8, 512
NORMS = ('SC', 'SH', 'SJ', 'SS')
CONDITIONS = ('none', 'one', 'multi')
CONDITION_LABELS = {'none': 'No inertia', 'one': 'One-step', 'multi': 'Multi-step'}
DEPTHS = {'SC': (2, 0), 'SH': (3, 0), 'SJ': (4, 3), 'SS': (1, 2)}
ARCHITECTURES = {'none': lambda n: (0, 0, 0., 0.), 'one': lambda n: (1, 1, 1., 1.), 'multi': lambda n: (*DEPTHS[n], 1., 1.)}
NAMES = {'SC': 'Scoring', 'SH': 'Shunning', 'SJ': 'Stern Judging', 'SS': 'Simple Standing'}
COLORS = {'none': '#6B6B6B', 'one': '#0077BB', 'multi': '#D55E00'}
LINEWIDTHS = {'none': .9, 'one': 1.2, 'multi': 1.5}
RATIOS = np.unique(np.r_[1.001, np.linspace(1.1, 7, 31), np.linspace(7.25, 10, 12), 5.])

## Data and local-protection thresholds

The scan uses the verified engine (all-good-selected reputations, exact evaluation wherever the reputation map is
not smooth; see `../engine/README.md`). Reputations do not depend on $b/c$, so the pure-DISC thresholds follow from
the resident reputation $g$ and the mutant reputations $g_{\rm ALLC}$, $g_{\rm ALLD}$:
$R_L=g/(g-g_{\rm ALLD})$ and $R_U=(1-g)/(g_{\rm ALLC}-g)$ (without inertia under Scoring these coincide).

In [2]:
scan = pd.read_csv(DATA / 'benefit_cost_scan.csv')
assert len(scan) == len(NORMS) * len(CONDITIONS) * len(RATIOS)
assert not scan.duplicated(['norm', 'condition', 'b_over_c']).any()
assert scan.n_initial.eq(N_SAMPLES * N_INITIAL).all()
for norm in NORMS:
    for cond in CONDITIONS:
        part = scan[(scan.norm == norm) & (scan.condition == cond)]
        np.testing.assert_allclose(np.sort(part.b_over_c), RATIOS, atol=1e-12)
        assert np.allclose(part[['k1', 'k2', 'alpha', 'beta']].iloc[0], ARCHITECTURES[cond](norm))
gap_failures = scan.loc[~scan.outcome_gap_ok, ['norm', 'condition', 'b_over_c', 'max_final_cooperation_defective',
                                             'min_final_cooperation_cooperative']]
print('Conditions without a clear outcome gap:', len(gap_failures))
display(gap_failures)
print('Step-cap hits:', scan.loc[scan.n_step_cap > 0].groupby(['norm', 'condition']).n_step_cap.sum().to_dict())

rules = []
for norm in NORMS:
    for cond in CONDITIONS:
        inv = analysis.pure_disc_invasion(norm, *ARCHITECTURES[cond](norm), b_over_c=5.0, e1=E1, e2=E2)
        g, gc, gd = inv['r_DISC_resident'], inv['r_ALLC_mutant'], inv['r_ALLD_mutant']
        lower = g / (g - gd) if g - gd > 1e-12 else np.inf
        upper = (1 - g) / (gc - g) if gc - g > 1e-12 else np.inf
        rules.append(dict(norm=norm, condition=cond, R_L=lower, R_U=upper, g_ALLC=gc, g_ALLD=gd, g_DISC=g,
                          resident_residual=inv['resident_residual']))
rules = pd.DataFrame(rules); rules.to_csv(DATA / 'invasion_thresholds.csv', index=False)
thresholds = rules.set_index(['norm', 'condition'])
display(rules.round(4))

Conditions without a clear outcome gap: 0
Empty DataFrame
Columns: [norm, condition, b_over_c, max_final_cooperation_defective, min_final_cooperation_cooperative]
Index: []
Step-cap hits: {('SC', 'none'): 9271, ('SC', 'one'): 40, ('SH', 'one'): 5, ('SJ', 'one'): 3, ('SS', 'multi'): 5, ('SS', 'one'): 6}
   norm condition      R_L         R_U  g_ALLC  g_ALLD  g_DISC  resident_residual
0    SC      none   1.0629      1.0629  0.9608  0.0200  0.3378                0.0
1    SC       one   1.0008      6.3070  0.9969  0.0008  0.9963                0.0
2    SC     multi   1.0625    230.2390  0.9999  0.0588  0.9999                0.0
3    SH      none  52.1267     52.1267  0.0392  0.0200  0.0204                0.0
4    SH       one   1.0008      5.6409  0.9961  0.0008  0.9953                0.0
5    SH     multi   1.0842   4410.3662  1.0000  0.0776  1.0000                0.0
6    SJ      none      inf         inf  0.5000  0.5000  0.5000                0.0
7    SJ       one   1.0012      5.6865  

## Panels

In [3]:
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 9,
                     'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.linewidth': .6, 'legend.fontsize': 8,
                     'legend.frameon': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})

def panel_curve(norm, cond):
    rows = scan[(scan.norm == norm) & (scan.condition == cond)].sort_values('b_over_c')
    x, mean, sem = (rows[k].to_numpy() for k in ('b_over_c', 'mean_cooperation', 'sem_mean_cooperation'))
    curve = PchipInterpolator(x, mean, extrapolate=False)
    return x, mean, sem, curve

def draw_panel(ax, norm, labelled):
    ax.axvline(5, color='#929292', lw=0.55, ls=(0, (3, 3)), zorder=0)
    for cond in CONDITIONS:
        color = COLORS[cond]; t = thresholds.loc[norm, cond]
        x, mean, sem, curve = panel_curve(norm, cond)
        boundaries = [v for v in (t.R_L, t.R_U) if x[0] <= v <= x[-1]]
        grid = np.unique(np.r_[np.linspace(x[0], x[-1], 1800), x, boundaries])
        y = curve(grid); spread = np.interp(grid, x, sem)
        ax.fill_between(grid, np.maximum(0, y - spread), np.minimum(1, y + spread), color=color, alpha=0.1, lw=0, zorder=1)
        cuts = np.unique(np.r_[x[0], boundaries, x[-1]])
        for start, stop in zip(cuts[:-1], cuts[1:]):
            segment = np.r_[start, grid[(grid > start) & (grid < stop)], stop]
            protected = t.R_L < (start + stop) / 2 < t.R_U
            ax.plot(segment, curve(segment), color=color, lw=LINEWIDTHS[cond], ls='-' if protected else (0, (3, 2)),
                    solid_capstyle='butt', dash_capstyle='butt', zorder=3 if protected else 2)
        if max(x[0], t.R_L) < min(x[-1], t.R_U) and t.R_U > 10:
            ax.plot(10, curve(10), marker='>', linestyle='none', color=color, ms=3.2, zorder=5, clip_on=False)
        if labelled and t.R_L < t.R_U and (x[0] < t.R_U < 10):
            offset = (5, 10) if cond == 'one' else (-5, 11)
            ax.annotate('$R_U$=' + f'{t.R_U:.2f}', (t.R_U, curve(t.R_U)), xytext=offset, textcoords='offset points',
                        fontsize=6, ha='left' if offset[0] > 0 else 'right', color='#333333',
                        arrowprops={'arrowstyle': '-', 'color': color, 'lw': 0.4})
    ax.set(xlim=(0.7, 10), ylim=(-0.03, 1.025), xticks=[1, 3, 5, 7, 10], yticks=[0, 0.5, 1])
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(direction='out', length=2.2, width=0.5, pad=3)
    if labelled:
        t = thresholds.loc[norm, 'none']
        if np.isfinite(t.R_L) and 1 < t.R_L < 10 and abs(t.R_L - t.R_U) < 1e-9:
            curve = panel_curve(norm, 'none')[-1]
            ax.annotate('$R_L=R_U$=' + f'{t.R_L:.2f}', (t.R_L, curve(t.R_L)), xytext=(2.35, 0.105), textcoords='data',
                        fontsize=5.7, color='#555555', arrowprops={'arrowstyle': '-', 'lw': 0.4, 'color': '#6B6B6B'})

def save_figure(fig, name):
    for extension in ('pdf', 'svg'):
        fig.savefig(FIGURES / f'{name}.{extension}', bbox_inches='tight', pad_inches=.04, facecolor='white')

def plot_panel(norm):
    fig, ax = plt.subplots(figsize=(4.1, 3.0)); fig.subplots_adjust(left=.18, right=.97, bottom=.23, top=.88)
    draw_panel(ax, norm, labelled=True)
    ax.set(title=NAMES[norm], xlabel=r'Benefit-to-cost ratio, $b/c$', ylabel=r'Mean cooperation $\bar\rho$')
    return fig

def plot_combined():
    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(9.8, 6.3))
    fig.subplots_adjust(left=.11, right=.985, bottom=.16, top=.80, wspace=.23, hspace=.42)
    for index, (norm, ax) in enumerate(zip(NORMS, axes.flat)):
        draw_panel(ax, norm, labelled=True)
        ax.set_title(NAMES[norm], loc='left', pad=6)
        ax.text(-.065, 1.065, 'abcd'[index], transform=ax.transAxes, fontsize=11, fontweight='bold', ha='right')
    handles = [Line2D([], [], color=COLORS[c], lw=LINEWIDTHS[c], ls=style, label=f'{CONDITION_LABELS[c]} ({state})')
               for style, state in [('-', 'protected'), ((0, (3, 2)), 'unprotected')] for c in CONDITIONS]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(.54, .995), ncol=2, handlelength=3,
               columnspacing=2.5, labelspacing=.7, frameon=False)
    fig.supylabel(r'Mean cooperation $\bar\rho$', x=.025, y=.48)
    fig.supxlabel(r'Benefit-to-cost ratio, $b/c$', x=.55, y=.07)
    fig.text(.55, .018, r'Solid: pure DISC locally protected | Dashed: not protected | Arrow: $R_U>10$ | Shading: $\pm$ s.e.m.',
             ha='center', fontsize=7, color='#555555')
    return fig

In [4]:
for norm in NORMS:
    fig = plot_panel(norm); save_figure(fig, f'panel_{norm}'); plt.close(fig)
fig = plot_combined(); save_figure(fig, 'Extended_Data_Figure4_combined'); plt.show(); plt.close(fig)
assert len(list(FIGURES.glob('*.pdf'))) == len(list(FIGURES.glob('*.svg'))) == 5
print('Saved four panels and one combined figure in PDF and SVG.')

Saved four panels and one combined figure in PDF and SVG.
